In [1]:
import pandas as pd
df = pd.read_csv('sample_kinder_data.csv')

In [2]:
def clean_data(df):
    # Drop rows with missing target values
    df = df.dropna(subset=['score_read', 'score_math'])

    # Drop 'ladder, 'schooldistrict_id'
    df = df.drop(columns=['ladder', 'schooldistrict_id'], errors='ignore')


    # Fix numeric column: experience
    df['experience'] = df['experience'].fillna(df['experience'].median())

    # Fix ethnicity BEFORE filling Unknown
    df['ethnicity'] = df['ethnicity'].apply(
        lambda x: x if x in ['cauc', 'afam'] else 'other'
    )

    # Replace missing categorical values
    df = df.fillna({
        'ethnicity': 'other',
        'lunch': 'Unknown',
        'birth': 'Unknown',
        'class_type': 'Unknown',
        't_ethnicity': 'Unknown',
        'degree': 'Unknown'
    })

    # Fix degree categories
    df['degree'] = df['degree'].apply(
        lambda x: x if x in ['bachelor', 'master'] else 'other'
    )
    return df


def remove_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return df[(df[col] >= lower) & (df[col] <= upper)]


df_clean = clean_data(df.copy())

for col in ['score_read', 'score_math', 'experience']:
    df_clean = remove_outliers_iqr(df_clean, col)

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer

X = df_clean.drop(columns=['score_read', 'score_math'])
y_read = df_clean['score_read']
y_math = df_clean['score_math']

X_train, X_test, y_read_train, y_read_test, y_math_train, y_math_test = train_test_split(
    X, y_read, y_math, test_size=0.2, random_state=42
)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

preprocessor = make_column_transformer(
    (StandardScaler(), num_cols),
    (OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
)

### grid search lightgbm

In [4]:
from sklearn.model_selection import GridSearchCV
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
import numpy as np

# -----------------------------
# Parameter grid (safe + small)
# -----------------------------
param_grid = {
    "model__n_estimators": [500, 1000, 2000],
    "model__learning_rate": [0.1, 0.05, 0.03],
    "model__num_leaves": [31, 63],
    "model__subsample": [1.0, 0.8],
    "model__colsample_bytree": [1.0, 0.8],
}

# ===========================================
# 1) GRID SEARCH FOR READING
# ===========================================
pipe_read = Pipeline([
    ("prep", preprocessor),
    ("model", LGBMRegressor(random_state=42))
])

grid_read = GridSearchCV(
    estimator=pipe_read,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=1
)

print("Running LightGBM Grid Search for READING...")
grid_read.fit(X_train, y_read_train)

print("\nBest params (READ):")
print(grid_read.best_params_)

print("\nBest CV MSE (READ):")
print(-grid_read.best_score_)

# Predict on test set with the best model
best_read_model = grid_read.best_estimator_
yhat_read_test = best_read_model.predict(X_test)
MSE_read_test_tuned = np.mean((yhat_read_test - y_read_test) ** 2)

print(f"\nTest MSE (READ, tuned): {MSE_read_test_tuned:.4f}")


# ===========================================
# 2) GRID SEARCH FOR MATH
# ===========================================
pipe_math = Pipeline([
    ("prep", preprocessor),
    ("model", LGBMRegressor(random_state=42))
])

grid_math = GridSearchCV(
    estimator=pipe_math,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=1
)

print("\n\nRunning LightGBM Grid Search for MATH...")
grid_math.fit(X_train, y_math_train)

print("\nBest params (MATH):")
print(grid_math.best_params_)

print("\nBest CV MSE (MATH):")
print(-grid_math.best_score_)

# Predict on test set with the best model
best_math_model = grid_math.best_estimator_
yhat_math_test = best_math_model.predict(X_test)
MSE_math_test_tuned = np.mean((yhat_math_test - y_math_test) ** 2)

print(f"\nTest MSE (MATH, tuned): {MSE_math_test_tuned:.4f}")


# ===========================================
# Combined MSE_o (OUTCOME FOR ASSIGNMENT)
# ===========================================
sq_err_read = (yhat_read_test - y_read_test) ** 2
sq_err_math = (yhat_math_test - y_math_test) ** 2

MSE_o_tuned = (sq_err_read + sq_err_math).sum() / (2 * len(X_test))

print(f"\nCombined MSE_o (tuned models): {MSE_o_tuned:.4f}")


Running LightGBM Grid Search for READING...
Fitting 3 folds for each of 72 candidates, totalling 216 fits


c:\Users\huiqi\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\huiqi\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\huiqi\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\huiqi\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000515 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 147
[LightGBM] [Info] Number of data points in the train set: 3570, number of used features: 23
[LightGBM] [Info] Start training from score 432.806723

Best params (READ):
{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__n_estimators': 500, 'model__num_leaves': 31, 'model__subsample': 1.0}

Best CV MSE (READ):
506.3913388529583

Test MSE (READ, tuned): 467.4188


Running LightGBM Grid Search for MATH...
Fitting 3 folds for each of 72 candidates, totalling 216 fits


c:\Users\huiqi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000461 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 147
[LightGBM] [Info] Number of data points in the train set: 3570, number of used features: 23
[LightGBM] [Info] Start training from score 481.867507

Best params (MATH):
{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__n_estimators': 500, 'model__num_leaves': 31, 'model__subsample': 1.0}

Best CV MSE (MATH):
1509.4544000830258

Test MSE (MATH, tuned): 1514.1838

Combined MSE_o (tuned models): 990.8013


c:\Users\huiqi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


### grid search random forest

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
import numpy as np

# -----------------------------
# Parameter grid (safe + effective)
# -----------------------------
rf_param_grid = {
    "mdl__n_estimators": [200, 500, 800],
    "mdl__max_depth": [5, 10, 15],
    "mdl__min_samples_split": [2, 5],
    "mdl__min_samples_leaf": [1, 2],
    "mdl__max_features": ["sqrt", "log2"],
}

# ==============================================================
# 1) GRID SEARCH FOR READING
# ==============================================================

pipe_read = Pipeline([
    ("prep", preprocessor),
    ("mdl", RandomForestRegressor(n_jobs=-1, random_state=42))
])

grid_read = GridSearchCV(
    estimator=pipe_read,
    param_grid=rf_param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=1
)

print("Running Random Forest Grid Search for READING...")
grid_read.fit(X_train, y_read_train)

# Best params + best CV score
print("\nBest params (READ):", grid_read.best_params_)
print("Best CV MSE (READ):", -grid_read.best_score_)

# Test MSE
best_read_model = grid_read.best_estimator_
yhat_read_test = best_read_model.predict(X_test)
MSE_read_test_tuned = np.mean((yhat_read_test - y_read_test)**2)
print(f"Test MSE (READ, tuned): {MSE_read_test_tuned:.4f}")


# ==============================================================
# 2) GRID SEARCH FOR MATH
# ==============================================================

pipe_math = Pipeline([
    ("prep", preprocessor),
    ("mdl", RandomForestRegressor(n_jobs=-1, random_state=42))
])

grid_math = GridSearchCV(
    estimator=pipe_math,
    param_grid=rf_param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=1
)

print("\nRunning Random Forest Grid Search for MATH...")
grid_math.fit(X_train, y_math_train)

# Best params + best CV score
print("\nBest params (MATH):", grid_math.best_params_)
print("Best CV MSE (MATH):", -grid_math.best_score_)

# Test MSE
best_math_model = grid_math.best_estimator_
yhat_math_test = best_math_model.predict(X_test)
MSE_math_test_tuned = np.mean((yhat_math_test - y_math_test)**2)
print(f"Test MSE (MATH, tuned): {MSE_math_test_tuned:.4f}")


# ==============================================================
# Combined MSE_o
# ==============================================================

sq_err_read = (yhat_read_test - y_read_test)**2
sq_err_math = (yhat_math_test - y_math_test)**2

MSE_o_tuned = (sq_err_read + sq_err_math).sum() / (2 * len(X_test))

print(f"\nCombined MSE_o (tuned RF models): {MSE_o_tuned:.4f}")


Running Random Forest Grid Search for READING...
Fitting 3 folds for each of 72 candidates, totalling 216 fits

Best params (READ): {'mdl__max_depth': 15, 'mdl__max_features': 'sqrt', 'mdl__min_samples_leaf': 2, 'mdl__min_samples_split': 2, 'mdl__n_estimators': 500}
Best CV MSE (READ): 539.9943416040675
Test MSE (READ, tuned): 537.4179

Running Random Forest Grid Search for MATH...
Fitting 3 folds for each of 72 candidates, totalling 216 fits
